In [2]:
extern crate onq;
use onq::{
    Circuit, CircuitBuilder, Operation, QduId, Simulator, SimulationResult, StableState, OnqError
};

fn qid(id: u64) -> QduId { QduId(id) }

// Define QDUs for the Quantum Attention Head
let input_token = qid(0); // The incoming sequence data
let q_reg = qid(1);       // Query representation
let k_reg = qid(2);       // Key representation
let v_reg = qid(3);       // Value representation
let ancilla = qid(4);     // Computes the attention score (inner product)

println!("Attention Head QDUs defined.");

Attention Head QDUs defined.


In [3]:
let mut builder = CircuitBuilder::new();
println!("Building Quantum Self-Attention Circuit using native onq patterns...");

// 1. Generate Q, K, and V Representations (The Transforms)
// Since we cannot use arbitrary strings, we build distinct mappings using derived patterns.
// Query Transform (W_Q proxy): Controlled-X
builder = builder.add_op(Operation::ControlledInteraction {
    control: input_token,
    target: q_reg,
    pattern_id: "QualityFlip".to_string(), 
});
// Key Transform (W_K proxy): Controlled-Z
builder = builder.add_op(Operation::ControlledInteraction {
    control: input_token,
    target: k_reg,
    pattern_id: "PhaseIntroduce".to_string(),
});
// Value Transform (W_V proxy): Superposition then Controlled-X
builder = builder.add_op(Operation::InteractionPattern {
    target: v_reg,
    pattern_id: "Superposition".to_string(),
});
builder = builder.add_op(Operation::ControlledInteraction {
    control: input_token,
    target: v_reg,
    pattern_id: "QualityFlip".to_string(),
});
println!("  Step 1: Q, K, and V registers generated using foundational patterns.");

// 2. Compute the Attention Score (Overlap of Q and K)
// We use a simplified interference mechanism using Controlled-QualityFlips
builder = builder.add_op(Operation::InteractionPattern {
    target: ancilla,
    pattern_id: "Superposition".to_string(),
});
builder = builder.add_op(Operation::ControlledInteraction {
    control: ancilla,
    target: q_reg,
    pattern_id: "QualityFlip".to_string(),
});
builder = builder.add_op(Operation::ControlledInteraction {
    control: ancilla,
    target: k_reg,
    pattern_id: "QualityFlip".to_string(),
});
builder = builder.add_op(Operation::InteractionPattern {
    target: ancilla,
    pattern_id: "Superposition".to_string(),
});
println!("  Step 2: Attention score encoded via native interference circuit.");

// 3. Modulate the Value Register
// The ancilla controls the phase of the Value register based on the score.
builder = builder.add_op(Operation::ControlledInteraction {
    control: ancilla,
    target: v_reg,
    pattern_id: "PhaseIntroduce".to_string(),
});
println!("  Step 3: Value register modulated via Controlled PhaseIntroduce.");

// 4. Stabilize the output
builder = builder.add_op(Operation::Stabilize {
    targets: vec![v_reg, ancilla],
});
println!("  Step 4: Output stabilization added.");

let circuit = builder.build();
println!("\nQuantum Self-Attention Circuit compiled.\n{}", circuit);

Building Quantum Self-Attention Circuit using native onq patterns...
  Step 1: Q, K, and V registers generated using foundational patterns.
  Step 2: Attention score encoded via native interference circuit.
  Step 3: Value register modulated via Controlled PhaseIntroduce.
  Step 4: Output stabilization added.

Quantum Self-Attention Circuit compiled.
onq::Circuit[10 operations on 5 QDUs]
QDU(0): ───@──────@─────────────@─────────────────────────────────────────────
           │      │             │                                             
QDU(1): ───X──────────────────────────────────X───────────────────────────────
                  │             │             │                               
QDU(2): ──────────●──────────────────────────────────X────────────────────────
                                │             │      │                        
QDU(3): ─────────────────H──────X──────────────────────────────────●──────M───
                                              │      │  

### Step 2: The Quantum Attention Score (Inner Product Estimation)

In a classical transformer, the attention score is derived from the dot product of the Query and Key vectors. In our quantum equivalent, this translates to finding the state overlap, or fidelity, between the $|Q\rangle$ and $|K\rangle$ registers.

We achieve this using an ancilla-assisted interference circuit (similar to a SWAP test). Here is the step-by-step mathematical evolution of the system:

**1. Initialization**
We begin with the ancilla QDU in the ground state, alongside our generated Query and Key states:
$$|\psi_0\rangle = |0\rangle_a |Q\rangle |K\rangle$$

**2. First Superposition**
Applying the first `Superposition` pattern (analogous to a Hadamard gate) puts the ancilla into an equal superposition:
$$|\psi_1\rangle = \frac{1}{\sqrt{2}}(|0\rangle_a + |1\rangle_a) |Q\rangle |K\rangle$$

**3. Controlled State Interference**
The `StateInterference_Q` and `StateInterference_K` operations act as a Controlled-SWAP. If the ancilla is $|1\rangle$, the states of the Query and Key registers are swapped. If $|0\rangle$, they remain unchanged:
$$|\psi_2\rangle = \frac{1}{\sqrt{2}}(|0\rangle_a |Q\rangle |K\rangle + |1\rangle_a |K\rangle |Q\rangle)$$

**4. Second Superposition (Re-interference)**
Applying the second `Superposition` pattern to the ancilla forces the states to interfere:
$$|\psi_3\rangle = \frac{1}{2}|0\rangle_a (|Q\rangle |K\rangle + |K\rangle |Q\rangle) + \frac{1}{2}|1\rangle_a (|Q\rangle |K\rangle - |K\rangle |Q\rangle)$$

**5. Measuring the Attention Score**
The probability of the ancilla resolving to the $|0\rangle$ state is directly tied to the inner product $\langle Q | K \rangle$. The probability distribution is defined as:
$$P(ancilla = 0) = \frac{1}{2}\left(1 + |\langle Q | K \rangle|^2\right)$$

In standard quantum mechanics, measuring this requires repeated shots to build a probability distribution. However, `onq`'s deterministic stabilization uses this pre-stabilized interference pattern to directly modulate the $|V\rangle$ (Value) register in the subsequent step, creating our non-linear attention mechanism without classical fallback.

In [4]:
// --- Run Simulation ---
let simulator = Simulator::new();
println!("\nRunning Quantum Attention simulation...");

match simulator.run(&circuit) {
    Ok(result) => {
        println!("Simulation finished successfully.");
        println!("\nSimulation Result Details:");
        println!("{}", result);

        println!("\n--- Analysis of Attention Mechanism ---");
        
        // Analyze Ancilla (Attention Score proxy)
        if let Some(StableState::ResolvedQuality(ancilla_outcome)) = result.get_stable_state(&ancilla) {
             println!("- Ancilla QDU ({}) stabilized to state: {}", ancilla, ancilla_outcome);
             println!("  (A '0' here suggests high overlap/fidelity between Q and K prior to stabilization)");
        } else {
             println!("- Ancilla QDU ({}) was not found in stabilization results.", ancilla);
        }
        
        // Analyze Value Register (Modulated Output)
        if let Some(StableState::ResolvedQuality(v_outcome)) = result.get_stable_state(&v_reg) {
             println!("- Value Register ({}) stabilized to state: {}", v_reg, v_outcome);
             println!("  (This represents the final 'contextualized' token state after non-linear modulation)");
        } else {
             println!("- Value Register ({}) was not found in stabilization results.", v_reg);
        }
        
        println!("\nNext steps for real-world application would involve scaling this to multi-head attention and observing sequence contextualization over multiple tokens.");
    }
    Err(e) => {
        eprintln!("\n--- Simulation Failed ---");
        eprintln!("Error: {}", e);
    }
}


Running Quantum Attention simulation...
Simulation finished successfully.

Simulation Result Details:
Simulation Results:
  Stable Outcomes:
    QDU(3): Stable(1)
    QDU(4): Stable(0)


--- Analysis of Attention Mechanism ---
- Ancilla QDU (QDU(4)) stabilized to state: 0
  (A '0' here suggests high overlap/fidelity between Q and K prior to stabilization)
- Value Register (QDU(3)) stabilized to state: 1
  (This represents the final 'contextualized' token state after non-linear modulation)

Next steps for real-world application would involve scaling this to multi-head attention and observing sequence contextualization over multiple tokens.


()